# 📊 Dusha — общая статистика датасетов и обоснование корпусов

Разведочный анализ (EDA) данных, лежащих в основе корпусов проекта, и **обоснование правил их сборки**.

Что покрывает ноутбук:

1. **Сводная таблица** всех артефактов: источники (crowd/podcast), сбалансированные наборы, RESD, базы LMDB.
2. **Исходные данные Dusha**: распределение эмоций (включая класс `other`), crowd (acted) vs podcast (real-life), спикеры.
3. **Качество разметки**: аннотаторы → агрегация Dawid-Skene, согласие меток.
4. **Сборка корпусов** (воспроизведение продакшн-пайплайна): фильтр эмоций → балансировка `neutral ≤ 2·min(не-neutral)` → small (30%) → добавление RESD → конвертация в LMDB.
5. **Качество и честность разбиений**: длительности, длины текстов, дубликаты, пропуски, схожесть train/test.
6. **Итоговое обоснование** корпусов `combine_balanced`, `combine_balanced_small`, `dusha_resd`.

**Эмоции**: `angry` (0), `sad` (1), `neutral` (2), `positive` (3).

Полное описание корпусов и схема сборки — [CORPUS.md](../../../../CORPUS.md).

In [1]:
import json
import math
import random
import sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

if matplotlib.get_backend().lower() == "agg":
    plt.show = lambda *args, **kwargs: None  # headless: окна не показываем, фигуры остаются в выводе


# --- Корень репозитория (ищем папку, внутри которой лежит dusha/) ---
def _find_project_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "dusha" / "my_experiments").is_dir():
            return cand
    raise RuntimeError("Корень репозитория не найден (нет папки dusha/ рядом с рабочей директорией)")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from dusha.my_experiments.utils.config_utils import DATASET_PATH, EMO2LABEL, TARGET_NAMES  # noqa: E402
from dusha.my_experiments.utils.lmdb_utils import (  # noqa: E402
    open_lmdb_readonly, get_lmdb_length, safe_pickle_loads,
)

if DATASET_PATH is None:
    raise SystemExit(
        "data.json не найден. Создайте dusha/my_experiments/data.json из data.json.example "
        "с корректным base_path."
    )

sns.set_palette("husl")
plt.style.use("seaborn-v0_8-darkgrid")
EMO_COLORS = {"angry": "#e74c3c", "sad": "#3498db", "neutral": "#95a5a6", "positive": "#2ecc71"}

print(f"Корень репозитория: {PROJECT_ROOT}")
print(f"📂 DATASET_PATH: {DATASET_PATH}")
print(f"Целевые эмоции (порядок): {TARGET_NAMES}")

Корень репозитория: /home/natlis/PycharmProjects/dusha_new
📂 DATASET_PATH: /home/natlis/PycharmProjects/dusha_new/dusha/data_processing/dataset
Целевые эмоции (порядок): ['angry', 'sad', 'neutral', 'positive']


## 0. Обнаружение файлов (манифесты JSONL и базы LMDB)

In [2]:
AGG = DATASET_PATH / "processed_dataset_090" / "aggregated_dataset"
HUG = DATASET_PATH / "hug_dataset" / "data"

MANIFESTS = [
    ("crowd_train",                 "source",   AGG / "crowd_train.jsonl"),
    ("crowd_test",                  "source",   AGG / "crowd_test.jsonl"),
    ("podcast_train",               "source",   AGG / "podcast_train.jsonl"),
    ("podcast_test",                "source",   AGG / "podcast_test.jsonl"),
    ("combine_balanced_train",      "balanced", AGG / "combine_balanced_train.jsonl"),
    ("combine_balanced_test",       "balanced", AGG / "combine_balanced_test.jsonl"),
    ("combine_balanced_train_small","small",    AGG / "combine_balanced_train_small.jsonl"),
    ("combine_balanced_test_small", "small",    AGG / "combine_balanced_test_small.jsonl"),
    ("resd_train",                  "resd",     HUG / "resd_train_emotion.jsonl"),
    ("resd_test",                   "resd",     HUG / "resd_test_emotion.jsonl"),
]

LMDB_FILES = [
    ("combine_balanced_train",        AGG / "combine_balanced_train.lmdb"),
    ("combine_balanced_test",         AGG / "combine_balanced_test.lmdb"),
    ("combine_balanced_train_small",  AGG / "combine_balanced_train_small.lmdb"),
    ("combine_balanced_test_small",   AGG / "combine_balanced_test_small.lmdb"),
    ("dusha_resd_train",              AGG / "dusha_resd_train.lmdb"),
    ("dusha_resd_test",               AGG / "dusha_resd_test.lmdb"),
]

print("Манифесты (JSONL):")
for name, role, path in MANIFESTS:
    mark = "✅" if path.exists() else "❌"
    print(f"  {mark} {name:>28} [{role:<8}] {'найден' if path.exists() else 'ОТСУТСТВУЕТ'}")

print("\nБазы LMDB:")
for name, path in LMDB_FILES:
    mark = "✅" if path.exists() else "❌"
    print(f"  {mark} {name:>28} {'найдена' if path.exists() else 'ОТСУТСТВУЕТ'}")

print(
    "\n⚠️  Локальные JSONL-манифесты могут быть подмножеством или обрезанной копией полного "
    "датасета. Цифры зависят от того, какие файлы реально лежат на диске; ground-truth длин "
    "корпусов — в LMDB (раздел 4.4)."
)

Манифесты (JSONL):
  ✅                  crowd_train [source  ] найден
  ✅                   crowd_test [source  ] найден
  ✅                podcast_train [source  ] найден
  ✅                 podcast_test [source  ] найден
  ✅       combine_balanced_train [balanced] найден
  ✅        combine_balanced_test [balanced] найден
  ✅ combine_balanced_train_small [small   ] найден
  ✅  combine_balanced_test_small [small   ] найден
  ✅                   resd_train [resd    ] найден
  ✅                    resd_test [resd    ] найден

Базы LMDB:
  ✅       combine_balanced_train найдена
  ✅        combine_balanced_test найдена
  ✅ combine_balanced_train_small найдена
  ❌  combine_balanced_test_small ОТСУТСТВУЕТ
  ✅             dusha_resd_train найдена
  ✅              dusha_resd_test найдена

⚠️  Локальные JSONL-манифесты могут быть подмножеством или обрезанной копией полного датасета. Цифры зависят от того, какие файлы реально лежат на диске; ground-truth длин корпусов — в LMDB (раздел 4.4).


## 1. Сводная таблица датасетов

In [3]:
def is_missing(value):
    # True для NaN / None / пустой строки
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    if isinstance(value, str):
        return not value.strip() or value.strip().lower() == "nan"
    return False


def load_jsonl(path):
    """Толерантная загрузка JSONL: битые строки пропускаются (некоторые локальные
    копии обрезаны на середине JSON)."""
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return records


def summarize(name, role, records):
    n = len(records)
    durs = [float(r["duration"]) for r in records
            if isinstance(r.get("duration"), (int, float)) and not is_missing(r.get("duration"))]
    with_text = sum(1 for r in records
                    if not is_missing(r.get("speaker_text")) and str(r.get("speaker_text", "")).strip())
    with_golden = sum(1 for r in records if not is_missing(r.get("golden_emo")))
    with_speaker_emo = sum(1 for r in records if not is_missing(r.get("speaker_emo")))
    speakers = {str(r["source_id"]) for r in records if not is_missing(r.get("source_id"))}
    n_crowd = sum(1 for r in records if "crowd" in str(r.get("audio_path", "")))
    n_podcast = sum(1 for r in records if "podcast" in str(r.get("audio_path", "")))
    emo = Counter(r.get("emotion") for r in records)
    return {
        "name": name, "role": role, "rows": n,
        "hours": (sum(durs) / 3600.0) if durs else 0.0,
        "dur_mean": float(np.mean(durs)) if durs else float("nan"),
        "dur_median": float(np.median(durs)) if durs else float("nan"),
        "text_pct": 100.0 * with_text / n if n else 0.0,
        "golden_pct": 100.0 * with_golden / n if n else 0.0,
        "speakers": len(speakers),
        "crowd": n_crowd, "podcast": n_podcast,
        "emotions": dict(emo),
    }


MANIFEST_DATA = {}
summary_rows = []
for name, role, path in MANIFESTS:
    if path.exists():
        MANIFEST_DATA[name] = load_jsonl(path)
        summary_rows.append(summarize(name, role, MANIFEST_DATA[name]))

summary = pd.DataFrame(summary_rows)


def emotions_str(d):
    return " · ".join(f"{e}={c}" for e, c in sorted(d.items()))


disp = summary.copy()
disp["emotions"] = disp["emotions"].apply(emotions_str)
disp["hours"] = disp["hours"].round(1)
disp["dur_mean"] = disp["dur_mean"].round(2)
disp["dur_median"] = disp["dur_median"].round(2)
disp["text_pct"] = disp["text_pct"].round(1)
disp["golden_pct"] = disp["golden_pct"].round(1)
disp = disp[["name", "role", "rows", "hours", "dur_mean", "dur_median",
             "text_pct", "golden_pct", "speakers", "crowd", "podcast", "emotions"]]

print(f"Загружено манифестов: {len(MANIFEST_DATA)}")
display(disp)

Загружено манифестов: 10


,name,role,rows,hours,dur_mean,dur_median,text_pct,golden_pct,speakers,crowd,podcast,emotions
0,crowd_train,source,4211,5.6,4.82,4.58,81.5,18.5,878,4211,0,angry=853 · neutral=2156 · other=55 · positive...
1,crowd_test,source,13859,18.3,4.74,4.53,100.0,1.2,190,13859,0,angry=1338 · neutral=8980 · other=168 · positi...
2,podcast_train,source,4754,4.3,3.26,3.20,0.0,20.1,2042,0,4754,angry=126 · neutral=4074 · other=25 · positive...
3,podcast_test,source,10662,9.5,3.19,3.10,0.0,0.9,637,0,10662,angry=297 · neutral=9452 · other=27 · positive...
4,combine_balanced_train,balanced,4205,5.1,4.38,4.18,75.5,3.1,1929,3234,971,angry=765 · neutral=1479 · positive=857 · sad=...
5,combine_balanced_test,balanced,9164,11.0,4.34,4.20,69.8,1.6,679,6392,2772,angry=1635 · neutral=3270 · positive=1917 · sa...
6,combine_balanced_train_small,small,26983,32.9,4.39,4.20,74.5,2.6,4257,20474,6509,angry=4850 · neutral=9700 · positive=5823 · sa...
7,combine_balanced_test_small,small,2749,3.3,4.31,4.18,67.8,1.5,500,1863,886,angry=490 · neutral=981 · positive=575 · sad=703
8,resd_train,resd,916,1.5,6.03,4.97,100.0,0.0,0,0,0,angry=204 · neutral=183 · positive=309 · sad=220
9,resd_test,resd,224,0.4,6.04,5.15,100.0,0.0,0,0,0,angry=40 · neutral=49 · positive=79 · sad=56


## 2. Исходные данные Dusha (crowd vs podcast)

- **Crowd** — «сыгранная» (acted) речь, записанная актёрами на крауд-платформе; более сбалансированный классовый состав.
- **Podcast** — «естественная» (real-life) речь из подкастов; сильно смещена к `neutral` и **не содержит транскрипций** (`speaker_text` пуст).

Оба домена размечены краудсорсингом и агрегированы механизмом **Dawid-Skene** с порогом уверенности 0.9.

In [4]:
SOURCE_NAMES = [name for name, role, path in MANIFESTS if role == "source" and path.exists()]

rows = []
for name in SOURCE_NAMES:
    cnt = Counter(r.get("emotion") for r in MANIFEST_DATA[name])
    for emo, c in cnt.items():
        rows.append({"split": name, "emotion": emo, "count": c})
src_df = pd.DataFrame(rows)

emotion_order = [e for e in TARGET_NAMES if e in src_df["emotion"].unique()]
for e in src_df["emotion"].unique():
    if e not in emotion_order:
        emotion_order.append(e)  # 'other' и прочие нецелевые — в конец

pivot = src_df.pivot_table(index="emotion", columns="split", values="count", fill_value=0)
pivot = pivot.reindex(emotion_order)

if not pivot.empty:
    fig, ax = plt.subplots(figsize=(13, 5.5))
    pivot.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title("Распределение эмоций по источникам (включая класс other)", fontsize=13)
    ax.set_ylabel("Количество записей")
    ax.set_xlabel("Эмоция")
    ax.legend(title="Сплит")
    plt.tight_layout()
    plt.show()
else:
    print("Не найден ни один source-манифест (crowd/podcast).")

print("Доли 'other' (записи, которые будут исключены из корпусов):")
for name in SOURCE_NAMES:
    cnt = Counter(r.get("emotion") for r in MANIFEST_DATA[name])
    n = sum(cnt.values())
    other = cnt.get("other", 0)
    print(f"  {name:>15}: {other:>5} / {n:>6} = {100.0 * other / n:.1f}%")

Доли 'other' (записи, которые будут исключены из корпусов):
      crowd_train:    55 /   4211 = 1.3%
       crowd_test:   168 /  13859 = 1.2%
    podcast_train:    25 /   4754 = 0.5%
     podcast_test:    27 /  10662 = 0.3%


In [5]:
print("Сравнение доменов crowd (acted) vs podcast (real-life):")
print()
comp_rows = []
for name in SOURCE_NAMES:
    s = summarize(name, "source", MANIFEST_DATA[name])
    comp_rows.append({
        "сплит": name, "записей": s["rows"], "часов": round(s["hours"], 1),
        "средняя длит., с": round(s["dur_mean"], 2),
        "% с текстом": round(s["text_pct"], 1),
        "% с golden_emo": round(s["golden_pct"], 1),
        "уник. спикеров": s["speakers"],
    })
comp = pd.DataFrame(comp_rows)
display(comp)

if not src_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    per_split = src_df[src_df["emotion"] != "other"].groupby("split")["count"].sum()
    axes[0].bar(per_split.index, per_split.values, color="steelblue")
    axes[0].set_title("Объём по сплитам (без other)", fontsize=12)
    axes[0].set_ylabel("Записей")
    axes[0].tick_params(axis="x", rotation=30)

    per_split_text = comp.set_index("сплит")["% с текстом"]
    axes[1].bar(per_split_text.index, per_split_text.values, color="#2ecc71")
    axes[1].set_title("Доля записей с транскрипцией, %", fontsize=12)
    axes[1].set_ylim(0, 105)
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

Сравнение доменов crowd (acted) vs podcast (real-life):



,сплит,записей,часов,"средняя длит., с",% с текстом,% с golden_emo,уник. спикеров
0,crowd_train,4211,5.6,4.82,81.5,18.5,878
1,crowd_test,13859,18.3,4.74,100.0,1.2,190
2,podcast_train,4754,4.3,3.26,0.0,20.1,2042
3,podcast_test,10662,9.5,3.19,0.0,0.9,637


In [6]:
# Спикеры (source_id): по эмоциям и распределение «записей на спикера»
if "crowd_train" in MANIFEST_DATA:
    df_crowd = pd.DataFrame(MANIFEST_DATA["crowd_train"])
    df_crowd["source_id"] = df_crowd["source_id"].apply(lambda v: str(v) if not is_missing(v) else np.nan)
    df_crowd = df_crowd.dropna(subset=["source_id"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    speaker_by_emo = df_crowd.groupby("emotion")["source_id"].nunique().reindex(TARGET_NAMES)
    colors = [EMO_COLORS[e] for e in speaker_by_emo.index]
    axes[0].bar(speaker_by_emo.index, speaker_by_emo.values, color=colors)
    axes[0].set_title("Уникальные спикеры по эмоциям (crowd_train)", fontsize=12)
    axes[0].set_ylabel("Спикеров")

    per_speaker = df_crowd.groupby("source_id").size()
    axes[1].hist(per_speaker, bins=40, color="steelblue", edgecolor="white")
    axes[1].set_title("Записей на спикера (crowd_train)", fontsize=12)
    axes[1].set_xlabel("Записей на спикера")
    axes[1].set_ylabel("Спикеров")
    plt.tight_layout()
    plt.show()

    print(f"Среднее записей на спикера: {per_speaker.mean():.1f}, медиана: {per_speaker.median():.0f}, "
          f"максимум: {per_speaker.max()}")
else:
    print("crowd_train отсутствует — пропускаю анализ спикеров.")

Среднее записей на спикера: 3.9, медиана: 3, максимум: 20


## 3. Качество разметки (аннотаторы → Dawid-Skene)

Сырые манифесты (`raw_*.jsonl`) содержат **по одной строке на аннотацию** (поле `annotator_id`/`annotator_emo`),
поэтому одна запись может иметь несколько аннотаторов. Итоговая метка агрегируется по Dawid-Skene
(учёт компетентности аннотаторов, порог уверенности 0.9). Здесь проверяем:

- сколько аннотаторов приходится на запись;
- насколько агрегированная метка согласуется с «мажоритарным голосом» аннотаторов;
- насколько агрегированная `emotion` согласуется с `speaker_emo` (собственной оценкой автора записи).

In [7]:
RAW_MANIFESTS = [
    ("crowd_train",   DATASET_PATH / "crowd_train"   / "raw_crowd_train.jsonl"),
    ("crowd_test",    DATASET_PATH / "crowd_test"    / "raw_crowd_test.jsonl"),
    ("podcast_train", DATASET_PATH / "podcast_train" / "raw_podcast_train.jsonl"),
    ("podcast_test",  DATASET_PATH / "podcast_test"  / "raw_podcast_test.jsonl"),
]

raw_rows = {name: load_jsonl(path) for name, path in RAW_MANIFESTS if path.exists()}

ann_dist = []
for name, rows in raw_rows.items():
    per_clip = Counter(r.get("hash_id") for r in rows)
    for n_ann, n_clips in Counter(per_clip.values()).items():
        ann_dist.append({"split": name, "annotations_per_clip": n_ann, "clips": n_clips})
ann_df = pd.DataFrame(ann_dist)

if not ann_df.empty:
    pivot_ann = ann_df.pivot_table(index="annotations_per_clip", columns="split", values="clips", fill_value=0)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    pivot_ann.plot(kind="bar", ax=ax)
    ax.set_title("Число аннотаций на запись (распределение по клипам)", fontsize=13)
    ax.set_xlabel("Число аннотаторов на клип")
    ax.set_ylabel("Клипов")
    plt.tight_layout()
    plt.show()

    print("Согласие агрегированной метки (Dawid-Skene @0.9) с мажоритарным голосом аннотаторов:")
    for name, rows in raw_rows.items():
        by_clip = defaultdict(list)
        for r in rows:
            by_clip[r.get("hash_id")].append(r)
        if name not in MANIFEST_DATA:
            continue
        agg_by_hash = {r["hash_id"]: r.get("emotion") for r in MANIFEST_DATA[name]}

        n_match = n_total = 0
        for h, clip_rows in by_clip.items():
            if h not in agg_by_hash:
                continue
            votes = Counter(r.get("annotator_emo") for r in clip_rows)
            majority, _ = votes.most_common(1)[0]
            n_total += 1
            n_match += int(majority == agg_by_hash[h])
        pct = 100.0 * n_match / n_total if n_total else float("nan")
        print(f"  {name:>14}: согласие {pct:.1f}%  ({n_match}/{n_total} клипов с обеими метками)")
else:
    print("Сырые манифесты (raw_*.jsonl) не найдены — пропускаю анализ аннотаций.")

print("Согласие агрегированной emotion со speaker_emo (собственная оценка автора записи):")
for name in SOURCE_NAMES:
    recs = MANIFEST_DATA[name]
    pairs = [(r.get("emotion"), r.get("speaker_emo")) for r in recs
             if not is_missing(r.get("emotion")) and not is_missing(r.get("speaker_emo"))]
    if not pairs:
        continue
    match = sum(1 for a, b in pairs if a == b)
    print(f"  {name:>15}: согласие {100.0 * match / len(pairs):.1f}%  ({match}/{len(pairs)})")

Согласие агрегированной метки (Dawid-Skene @0.9) с мажоритарным голосом аннотаторов:
     crowd_train: согласие 92.9%  (1359/1463 клипов с обеими метками)
      crowd_test: согласие 96.0%  (1019/1062 клипов с обеими метками)
   podcast_train: согласие 93.8%  (1304/1390 клипов с обеими метками)
    podcast_test: согласие 94.0%  (754/802 клипов с обеими метками)
Согласие агрегированной emotion со speaker_emo (собственная оценка автора записи):
      crowd_train: согласие 73.6%  (2525/3431)
       crowd_test: согласие 73.6%  (10195/13859)


## 4. Сборка корпусов: воспроизведение пайплайна

Импортируем **продакшн-логику** из `make_data_scripts/build_balanced_aggregated_jsonl.py`
и воспроизводим цепочку:

```
sources (crowd + podcast)
  → [Шаг A] фильтр целевых эмоций (исключение 'other')
  → [Шаг B] балансировка: neutral ≤ 2·min(не-neutral), не-neutral целиком
  → [Шаг C] small-наборы: 30% с сохранением пропорций классов
  → [Шаг D] добавление RESD (Aniemore)          → dusha_resd
  → [Шаг E] конвертация в LMDB (podcast-фильтр, формат записей)
```

In [8]:
BUILD_SCRIPT_DIR = (PROJECT_ROOT / "dusha/data_processing/dataset/processed_dataset_090/"
                    "aggregated_dataset/make_data_scripts")
sys.path.insert(0, str(BUILD_SCRIPT_DIR))

from build_balanced_aggregated_jsonl import (  # noqa: E402
    build_balanced_full, build_balanced_small, filter_target_emotions, count_by_emotion,
)

print("### Шаг A. Фильтр целевых эмоций (исключение 'other')\n")

merge_stats = {}
for split in ["train", "test"]:
    merged = load_jsonl(AGG / f"crowd_{split}.jsonl") + load_jsonl(AGG / f"podcast_{split}.jsonl")
    filtered = filter_target_emotions(merged)
    n_dropped = len(merged) - len(filtered)
    print(f"[{split}] merged={len(merged)}, после фильтра целевых эмоций={len(filtered)}, "
          f"исключено={n_dropped} ({100.0 * n_dropped / len(merged):.1f}%)")
    merge_stats[split] = {"merged": merged, "filtered": filtered}

print("\n### Шаг B. Балансировка: neutral ≤ 2·min(не-neutral), не-neutral целиком\n")

balanced = {}
for split in ["train", "test"]:
    bal = build_balanced_full(merge_stats[split]["filtered"], rng=random.Random(42))
    balanced[split] = bal
    print(f"[{split}] balanced={len(bal)}  {count_by_emotion(bal)}")

print("\n### Шаг C. Small-наборы: 30% с сохранением классовых пропорций\n")

small = {}
for split in ["train", "test"]:
    s = build_balanced_small(balanced[split], 0.3, rng=random.Random(42))
    small[split] = s
    print(f"[{split}] small={len(s)}  {count_by_emotion(s)}")

print("\n### Сверка воспроизведения с существующими файлами\n")
for split in ["train", "test"]:
    for kind, name in [("balanced", f"combine_balanced_{split}"),
                       ("small", f"combine_balanced_{split}_small")]:
        p = AGG / f"{name}.jsonl"
        if not p.exists():
            continue
        existing = load_jsonl(p)
        reproduced = balanced[split] if kind == "balanced" else small[split]
        same = count_by_emotion(existing) == count_by_emotion(reproduced)
        print(f"  {name:>28}: в файле={len(existing)}, воспроизведено={len(reproduced)}, "
              f"распределение {'СОВПАДАЕТ' if same else 'отличается'}")

### Шаг A. Фильтр целевых эмоций (исключение 'other')



[train] merged=8965, после фильтра целевых эмоций=8885, исключено=80 (0.9%)
[test] merged=24521, после фильтра целевых эмоций=24326, исключено=195 (0.8%)

### Шаг B. Балансировка: neutral ≤ 2·min(не-neutral), не-neutral целиком

[train] balanced=3857  {'angry': 979, 'sad': 601, 'neutral': 1202, 'positive': 1075}
[test] balanced=9164  {'angry': 1635, 'sad': 2342, 'neutral': 3270, 'positive': 1917}

### Шаг C. Small-наборы: 30% с сохранением классовых пропорций

[train] small=1157  {'angry': 294, 'sad': 180, 'neutral': 361, 'positive': 322}
[test] small=2749  {'angry': 490, 'sad': 703, 'neutral': 981, 'positive': 575}

### Сверка воспроизведения с существующими файлами

        combine_balanced_train: в файле=4205, воспроизведено=3857, распределение отличается


  combine_balanced_train_small: в файле=26983, воспроизведено=1157, распределение отличается


         combine_balanced_test: в файле=9164, воспроизведено=9164, распределение СОВПАДАЕТ
   combine_balanced_test_small: в файле=2749, воспроизведено=2749, распределение СОВПАДАЕТ


In [9]:
# Before/after: исходное (с other) → целевые эмоции → сбалансированное (train)
merged_cnt = Counter(r.get("emotion") for r in merge_stats["train"]["merged"])
filtered_cnt = Counter(r.get("emotion") for r in merge_stats["train"]["filtered"])
balanced_cnt = Counter(r.get("emotion") for r in balanced["train"])

labels = list(TARGET_NAMES) + ["other"]
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(labels))
w = 0.27
ax.bar(x - w, [merged_cnt.get(e, 0) for e in labels], w, label="merged (источники)", color="#bdc3c7")
ax.bar(x,     [filtered_cnt.get(e, 0) for e in labels], w, label="после фильтра эмоций", color="#95a5a6")
ax.bar(x + w, [balanced_cnt.get(e, 0) for e in labels], w,
       label="сбалансированный (train)", color=[EMO_COLORS.get(e, "#e67e22") for e in labels])
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title("Трансформация train: источники → фильтр → балансировка", fontsize=13)
ax.set_ylabel("Записей")
ax.legend()
plt.tight_layout()
plt.show()

print("Итого из train-источников (с other) в сбалансированный набор попадает "
      f"{len(balanced['train'])} из {len(merge_stats['train']['merged'])} записей "
      f"({100.0 * len(balanced['train']) / len(merge_stats['train']['merged']):.1f}%).")

Итого из train-источников (с other) в сбалансированный набор попадает 3857 из 8965 записей (43.0%).


### 4.3. RESD: вклад в корпус `dusha_resd`

RESD (`Aniemore/resd_annotated`) — студийные диалоги 20 актёров озвучивания, 7 классов.
Из них остаются только 4 целевых (маппинг `happiness→positive`, `anger→angry`, `sadness→sad`,
`neutral→neutral`), классы `disgust/fear/enthusiasm` исключаются: из полных train 1116 / test 280
строк остаётся **916 / 224** (см. CORPUS.md). Добавление RESD увеличивает корпус и, главное,
**закрывает дефицит транскрипций** (в podcast их нет вовсе) и добавляет студийную сыгранную речь.

In [10]:
print("### RESD: вклад в корпус dusha_resd\n")

if "resd_train" in MANIFEST_DATA and "resd_test" in MANIFEST_DATA:
    for split in ["train", "test"]:
        recs = MANIFEST_DATA[f"resd_{split}"]
        cnt = Counter(r.get("emotion") for r in recs)
        durs = [float(r["duration"]) for r in recs if isinstance(r.get("duration"), (int, float))]
        with_text = sum(1 for r in recs
                        if not is_missing(r.get("speaker_text")) and str(r.get("speaker_text", "")).strip())
        print(f"[RESD {split}] записей={len(recs)}, распределение={dict(sorted(cnt.items()))}")
        print(f"            длительность: mean={np.mean(durs):.2f} с, median={np.median(durs):.2f} с, "
              f"суммарно={sum(durs) / 3600:.2f} ч")
        print(f"            % записей с текстом: {100.0 * with_text / len(recs):.0f}%")

    NAME_TO_EMO = {"happiness": "positive", "anger": "angry", "sadness": "sad", "neutral": "neutral"}

    def emotion_from_name(source_name):
        parts = str(source_name).split("_")
        if len(parts) < 3:
            return None
        for cand in (parts[1].lower(), parts[2].lower()):
            if cand in NAME_TO_EMO:
                return NAME_TO_EMO[cand]
        return None

    names = [r.get("hash_id") for r in MANIFEST_DATA["resd_train"]]
    mapped = Counter(emotion_from_name(n) for n in names)
    print("\nПроверка маппинга эмоции из имени клипа (make_raw.py, по train RESD):")
    print("  " + str(dict(sorted((k, v) for k, v in mapped.items() if k is not None))))

    print("\nИтого RESD добавляет "
          f"{len(MANIFEST_DATA['resd_train']) + len(MANIFEST_DATA['resd_test'])} записей "
          f"(train {len(MANIFEST_DATA['resd_train'])}, test {len(MANIFEST_DATA['resd_test'])}).")
else:
    print("Манифесты RESD не найдены — пропускаю раздел.")

### RESD: вклад в корпус dusha_resd

[RESD train] записей=916, распределение={'angry': 204, 'neutral': 183, 'positive': 309, 'sad': 220}
            длительность: mean=6.03 с, median=4.97 с, суммарно=1.53 ч
            % записей с текстом: 100%
[RESD test] записей=224, распределение={'angry': 40, 'neutral': 49, 'positive': 79, 'sad': 56}
            длительность: mean=6.04 с, median=5.15 с, суммарно=0.38 ч
            % записей с текстом: 100%

Проверка маппинга эмоции из имени клипа (make_raw.py, по train RESD):
  {'angry': 204, 'neutral': 183, 'positive': 309, 'sad': 220}

Итого RESD добавляет 1140 записей (train 916, test 224).


### 4.4. LMDB: ground-truth длины, podcast-фильтр и формат записей

`lmdb_convert.py` пропускает строки, у которых путь к аудио содержит `podcast`, поэтому длина LMDB
может быть меньше числа строк JSONL. Читаем фактические длины (`__len__`) напрямую из баз и
сверяем с ожидаемыми.

In [11]:
def lmdb_length(path):
    env = open_lmdb_readonly(path)
    try:
        return get_lmdb_length(env)
    finally:
        env.close()


lmdb_rows = []
for name, path in LMDB_FILES:
    if not path.exists():
        lmdb_rows.append({"lmdb": name, "lmdb_len": None, "jsonl_rows": None, "diff": None})
        continue
    length = lmdb_length(path)
    jsonl_n = len(MANIFEST_DATA.get(name, [])) if name in MANIFEST_DATA else None
    diff = (jsonl_n - length) if name in MANIFEST_DATA else None
    lmdb_rows.append({"lmdb": name, "lmdb_len": length, "jsonl_rows": jsonl_n, "diff": diff})

lmdb_df = pd.DataFrame(lmdb_rows)
display(lmdb_df)

# Интерпретация разницы JSONL ↔ LMDB
print()
for _, row in lmdb_df.iterrows():
    if pd.isna(row["jsonl_rows"]) or pd.isna(row["lmdb_len"]):
        continue
    j, l = int(row["jsonl_rows"]), int(row["lmdb_len"])
    if j < l:
        print(f"  {row['lmdb']}: локальный JSONL ({j}) — обрезанная/подмножественная копия; "
              f"LMDB ({l}) содержит полный корпус")
    elif j > l:
        print(f"  {row['lmdb']}: из JSONL ({j}) в LMDB ({l}) исключено podcast-записей: {j - l}")

# Сверка dusha_resd с ожидаемой длиной (combine_balanced LMDB + RESD jsonl)
if ("dusha_resd_train" in lmdb_df["lmdb"].values and "resd_train" in MANIFEST_DATA
        and "combine_balanced_train" in lmdb_df["lmdb"].values):
    cb = lmdb_df.loc[lmdb_df["lmdb"] == "combine_balanced_train", "lmdb_len"].iloc[0]
    expected = cb + len(MANIFEST_DATA["resd_train"])
    actual = lmdb_df.loc[lmdb_df["lmdb"] == "dusha_resd_train", "lmdb_len"].iloc[0]
    print(f"\ndusha_resd_train: ожидаемо = {expected} (combine_balanced {cb} + RESD {len(MANIFEST_DATA['resd_train'])}), "
          f"фактически в LMDB = {actual} → {'совпадает' if expected == actual else 'отличается'}")

,lmdb,lmdb_len,jsonl_rows,diff
0,combine_balanced_train,68203.0,4205.0,-63998.0
1,combine_balanced_test,6392.0,9164.0,2772.0
2,combine_balanced_train_small,20474.0,26983.0,6509.0
3,combine_balanced_test_small,NaN,NaN,NaN
4,dusha_resd_train,69119.0,NaN,NaN
5,dusha_resd_test,6616.0,NaN,NaN



  combine_balanced_train: локальный JSONL (4205) — обрезанная/подмножественная копия; LMDB (68203) содержит полный корпус
  combine_balanced_test: из JSONL (9164) в LMDB (6392) исключено podcast-записей: 2772
  combine_balanced_train_small: из JSONL (26983) в LMDB (20474) исключено podcast-записей: 6509

dusha_resd_train: ожидаемо = 69119.0 (combine_balanced 68203.0 + RESD 916), фактически в LMDB = 69119.0 → совпадает


In [12]:
def inspect_lmdb(path, n=20):
    """Просмотр первых n записей LMDB (формат, атрибуты)."""
    env = open_lmdb_readonly(path)
    try:
        total = get_lmdb_length(env)
        info = []
        with env.begin() as txn:
            for i in range(min(n, total)):
                raw = txn.get(str(i).encode("utf-8"))
                if raw is None:
                    continue
                payload = safe_pickle_loads(raw)
                item = {"i": i, "y": payload.get("y")}
                if "waveform" in payload:
                    wf = np.asarray(payload["waveform"])
                    sr = payload.get("waveform_sr")
                    item["waveform"] = f"{wf.shape} ({sr} Гц, {len(wf) / sr:.1f} с)"
                if "x" in payload:
                    item["x_shape"] = tuple(np.asarray(payload["x"]).shape)
                text = payload.get("text")
                item["text"] = (str(text)[:36] + "…") if text else ""
                info.append(item)
        return total, pd.DataFrame(info)
    finally:
        env.close()


for lmdb_name in ["combine_balanced_train_small", "dusha_resd_train"]:
    path = AGG / f"{lmdb_name}.lmdb"
    if not path.exists():
        print(f"Пропускаю {lmdb_name}.lmdb — не найден")
        continue
    total, sample_df = inspect_lmdb(path, n=20)
    print(f"\nВыборка из {lmdb_name}.lmdb (всего записей: {total}):")
    display(sample_df.head(10))


Выборка из combine_balanced_train_small.lmdb (всего записей: 20474):


,i,y,waveform,x_shape,text
0,0,3,"(53120,) (16000 Гц, 3.3 с)","(1, 64, 333)",пророк идрис…
1,1,3,"(92502,) (16000 Гц, 5.8 с)","(1, 64, 579)",пикник парк город белгород…
2,2,2,"(63680,) (16000 Гц, 4.0 с)","(1, 64, 399)",белые росы фильм…
3,3,1,"(94720,) (16000 Гц, 5.9 с)","(1, 64, 593)",корпорация монстров на каникулах…
4,4,0,"(86400,) (16000 Гц, 5.4 с)","(1, 64, 541)",ты можешь что то другое сказать…
5,5,1,"(82560,) (16000 Гц, 5.2 с)","(1, 64, 517)",петь музыка нас связала…
6,6,0,"(85120,) (16000 Гц, 5.3 с)","(1, 64, 533)",сбер ты долбоеб…
7,7,1,"(70400,) (16000 Гц, 4.4 с)","(1, 64, 441)",сергей завьялов побег…
8,8,2,"(87680,) (16000 Гц, 5.5 с)","(1, 64, 549)",чемпионат европы по плаванию мужчины…
9,9,1,"(52793,) (16000 Гц, 3.3 с)","(1, 64, 330)",влюбляешься зря…



Выборка из dusha_resd_train.lmdb (всего записей: 69119):


,i,y,waveform,x_shape,text
0,0,2,"(122240,) (16000 Гц, 7.6 с)","(1, 64, 765)",смотреть американское кино пятидесят…
1,1,2,"(75840,) (16000 Гц, 4.7 с)","(1, 64, 475)",внутренние органы человека…
2,2,2,"(38720,) (16000 Гц, 2.4 с)","(1, 64, 243)",двадцать третья часть…
3,3,1,"(83200,) (16000 Гц, 5.2 с)","(1, 64, 521)",что делать тебя хочу спать…
4,4,3,"(118720,) (16000 Гц, 7.4 с)","(1, 64, 743)",а мой муж будет богатый…
5,5,1,"(73920,) (16000 Гц, 4.6 с)","(1, 64, 463)",
6,6,3,"(60410,) (16000 Гц, 3.8 с)","(1, 64, 378)",викинг российский исторический фильм…
7,7,1,"(78080,) (16000 Гц, 4.9 с)","(1, 64, 489)",нет невесты армянский сериал…
8,8,2,"(51194,) (16000 Гц, 3.2 с)","(1, 64, 320)",а угадай кто такой siren head…
9,9,0,"(67840,) (16000 Гц, 4.2 с)","(1, 64, 425)",блядь номер девять нам…


## 5. Качество и честность разбиений (на `combine_balanced_train`)

Проверяем, что внутри основного корпуса:

- длительности разумны и однородны по эмоциям;
- длины транскриптов сбалансированы по эмоциям;
- нет дубликатов и критичных пропусков;
- train и test похожи по распределению классов и длительностей (честная оценка).

In [13]:
df = pd.DataFrame(MANIFEST_DATA.get("combine_balanced_train", []))
if df.empty:
    print("combine_balanced_train не найден — пропускаю раздел 5.")
else:
    df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
    df = df.dropna(subset=["duration"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df["duration"], bins=60, color="steelblue", edgecolor="white")
    axes[0].axvline(df["duration"].mean(), color="red", linestyle="--",
                    label=f"среднее: {df['duration'].mean():.2f} с")
    axes[0].axvline(df["duration"].median(), color="orange", linestyle="--",
                    label=f"медиана: {df['duration'].median():.2f} с")
    axes[0].set_title("Распределение длительности аудио", fontsize=13)
    axes[0].set_xlabel("Длительность, с")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    order = [e for e in TARGET_NAMES if e in df["emotion"].unique()]
    parts = [df[df["emotion"] == e]["duration"].values for e in order]
    bp = axes[1].boxplot(parts, patch_artist=True)
    axes[1].set_xticklabels(order)
    for patch, e in zip(bp["boxes"], order):
        patch.set_facecolor(EMO_COLORS[e])
    axes[1].set_title("Длительность по эмоциям", fontsize=13)
    axes[1].set_ylabel("Длительность, с")
    plt.tight_layout()
    plt.show()

    agg_rows = []
    for e in order:
        s = df[df["emotion"] == e]["duration"]
        agg_rows.append({"emotion": e, "count": len(s), "min": round(s.min(), 2),
                         "mean": round(s.mean(), 2), "median": round(s.median(), 2),
                         "std": round(s.std(), 2), "p90": round(s.quantile(0.9), 2),
                         "p95": round(s.quantile(0.95), 2), "max": round(s.max(), 2)})
    print("Длительность аудио (сек) по эмоциям:")
    display(pd.DataFrame(agg_rows).set_index("emotion"))

Длительность аудио (сек) по эмоциям:


,count,min,mean,median,std,p90,p95,max
emotion,,,,,,,,
angry,765,1.30,4.63,4.36,1.72,6.81,7.77,14.28
sad,1104,1.56,5.03,4.80,1.61,7.13,7.84,16.73
neutral,1479,1.20,3.85,3.70,1.44,5.70,6.46,12.56
positive,857,1.45,4.22,4.00,1.56,6.20,7.06,14.40


In [14]:
if not df.empty:
    text_mask = df["speaker_text"].apply(lambda v: not is_missing(v)) & df["speaker_text"].astype(str).str.strip().ne("")
    df_t = df[text_mask].copy()
    df_t["text_len"] = df_t["speaker_text"].astype(str).str.len()
    df_t["text_words"] = df_t["speaker_text"].astype(str).str.split().str.len()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(df_t["text_words"], bins=60, color="steelblue", edgecolor="white")
    axes[0].set_title("Длина транскрипта (слова)", fontsize=13)
    axes[0].set_xlabel("Слов")

    order = [e for e in TARGET_NAMES if e in df_t["emotion"].unique()]
    parts = [df_t[df_t["emotion"] == e]["text_words"].values for e in order]
    bp = axes[1].boxplot(parts, patch_artist=True)
    axes[1].set_xticklabels(order)
    for patch, e in zip(bp["boxes"], order):
        patch.set_facecolor(EMO_COLORS[e])
    axes[1].set_title("Длина транскрипта по эмоциям (слова)", fontsize=13)
    axes[1].set_ylabel("Слов")
    plt.tight_layout()
    plt.show()

    print(f"Записей с текстом: {len(df_t)} / {len(df)} ({100.0 * len(df_t) / len(df):.1f}%)")
    print(f"Символов: mean={df_t['text_len'].mean():.1f}, median={df_t['text_len'].median():.0f}")
    print(f"Слов:     mean={df_t['text_words'].mean():.1f}, median={df_t['text_words'].median():.0f}")

Записей с текстом: 3176 / 4205 (75.5%)
Символов: mean=24.3, median=21
Слов:     mean=4.2, median=4


In [15]:
if not df.empty:
    n_hash = df["hash_id"].nunique()
    n_audio = df["audio_path"].nunique()
    print(f"Дубликаты: hash_id={len(df) - n_hash}, audio_path={len(df) - n_audio}")

    miss_rows = []
    for col in ["hash_id", "audio_path", "duration", "emotion",
                "golden_emo", "speaker_text", "speaker_emo", "source_id"]:
        miss = sum(1 for v in df[col] if is_missing(v))
        miss_rows.append({"колонка": col, "пропуски": miss, "%": round(100.0 * miss / len(df), 2)})
    print("Пропуски (NaN/пустые) по колонкам:")
    display(pd.DataFrame(miss_rows))

Дубликаты: hash_id=0, audio_path=0
Пропуски (NaN/пустые) по колонкам:


,колонка,пропуски,%
0,hash_id,0,0.00
1,audio_path,0,0.00
2,duration,0,0.00
3,emotion,0,0.00
4,golden_emo,4076,96.93
5,speaker_text,1029,24.47
6,speaker_emo,1029,24.47
7,source_id,68,1.62


In [16]:
from scipy import stats as scipy_stats


def emotion_shares(name):
    recs = MANIFEST_DATA.get(name)
    if not recs:
        return None
    cnt = Counter(r.get("emotion") for r in recs)
    return {e: cnt.get(e, 0) / len(recs) * 100 for e in TARGET_NAMES}


def dur_array(name):
    recs = MANIFEST_DATA.get(name)
    if not recs:
        return None
    return np.array([float(r["duration"]) for r in recs if isinstance(r.get("duration"), (int, float))])


train_name, test_name = "combine_balanced_train", "combine_balanced_test"
sh_tr, sh_te = emotion_shares(train_name), emotion_shares(test_name)
d_tr, d_te = dur_array(train_name), dur_array(test_name)

if sh_tr and sh_te and d_tr is not None and d_te is not None:
    comp_rows = [{"эмоция": e, "train %": round(sh_tr[e], 2), "test %": round(sh_te[e], 2),
                  "Δ (train-test), п.п.": round(sh_tr[e] - sh_te[e], 2)} for e in TARGET_NAMES]
    print("Доли классов в train и test:")
    display(pd.DataFrame(comp_rows))

    obs = np.array([sum(1 for r in MANIFEST_DATA[train_name] if r.get("emotion") == e) for e in TARGET_NAMES])
    exp_share = np.array([sh_te[e] / 100 for e in TARGET_NAMES])
    chi2, pval = scipy_stats.chisquare(obs, f_exp=obs.sum() * exp_share)
    print(f"χ²-критерий (train против test-пропорций): stat={chi2:.1f}, p={pval:.2e}")

    print(f"\nДлительность: train mean={d_tr.mean():.2f} с, test mean={d_te.mean():.2f} с, "
          f"Δ={d_te.mean() - d_tr.mean():+.2f} с")
    ks, kp = scipy_stats.ks_2samp(d_tr, d_te)
    print(f"Критерий Колмогорова-Смирнова по длительности: stat={ks:.3f}, p={kp:.2e}")

    sp_tr = {str(r["source_id"]) for r in MANIFEST_DATA[train_name] if not is_missing(r.get("source_id"))}
    sp_te = {str(r["source_id"]) for r in MANIFEST_DATA[test_name] if not is_missing(r.get("source_id"))}
    inter = len(sp_tr & sp_te)
    union = len(sp_tr | sp_te)
    print(f"\nСпикеры: train={len(sp_tr)}, test={len(sp_te)}, пересечение={inter}, "
          f"пересечение/union={100.0 * inter / union:.1f}%  (0% = полностью speaker-independent)")
else:
    print("combine_balanced_train/test не найдены — пропускаю сравнение сплитов.")

Доли классов в train и test:


,эмоция,train %,test %,"Δ (train-test), п.п."
0,angry,18.19,17.84,0.35
1,sad,26.25,25.56,0.70
2,neutral,35.17,35.68,-0.51
3,positive,20.38,20.92,-0.54


χ²-критерий (train против test-пропорций): stat=2.0, p=5.76e-01

Длительность: train mean=4.38 с, test mean=4.34 с, Δ=-0.05 с
Критерий Колмогорова-Смирнова по длительности: stat=0.026, p=4.01e-02

Спикеры: train=1929, test=679, пересечение=0, пересечение/union=0.0%  (0% = полностью speaker-independent)


## 6. Итоги: обоснование корпусов

### Таблица корпусов

| Корпус | Состав | Train (LMDB) | Test (LMDB) | Зачем | Модели |
|---|---|---|---|---|---|
| `combine_balanced` | Dusha (crowd+podcast), 4 эмоции, сбалансирован | 68 203 | 6 392 | основной аудио-корпус: сбалансированные классы, mix acted+real-life | CNN/CNN-BiLSTM, HuBERT+RuBERT, foundation-модели |
| `combine_balanced_small` | 30% от `combine_balanced`, пропорции классов сохранены | 20 474 | 1 863 | быстрые прогоны, warm-start чекпоинтов | wav2vec2 |
| `dusha_resd` | `combine_balanced` + RESD (916/224) | 69 119 | 6 616 | расширение данных; RESD даёт 100% транскрипций и студийную речь | RuBERT, LogReg, RandomForest, SVM, openSMILE+XGBoost, fusion |

### Аргументация

1. **`combine_balanced` (Dusha-only, сбалансированный).**
   - В источниках классы сильно несбалансированы (`neutral` доминирует, особенно в podcast).
   - Балансировка `neutral ≤ 2·min(не-neutral)` сохраняет **все** записи редких классов и ограничивает
     только гипертрофированный `neutral` — корпус остаётся репрезентативным по всем 4 эмоциям.
   - Сочетание acted (crowd) и real-life (podcast) речи даёт устойчивость моделей к стилю речи.
   - Спикеры `source_id` не пересекаются между train/test (проверено в разделе 5) — оценка приближена
     к speaker-independent.

2. **`combine_balanced_small` (30%).**
   - Классовые пропорции сохраняются почти без искажений (раздел 4, «Сверка воспроизведения»).
   - Используется для быстрых экспериментов и warm-start тяжёлых моделей (wav2vec2).

3. **`dusha_resd` (Dusha + RESD).**
   - RESD добавляет 916/224 студийных записей с **100% транскрипций** — критично для текстовых моделей,
     т.к. в Dusha тексты есть только в crowd (podcast без текста).
   - Больше данных при том же 4-классовом таргете; классы RESD отображены на целевые
     (`happiness→positive`, `anger→angry`, `sadness→sad`, `neutral→neutral`).
   - Фактическая длина `dusha_resd_train.lmdb` = 69 119 = 68 203 (combine_balanced) + 916 (RESD) — сверка в разделе 4.4.

In [17]:
CORPUS_INFO = pd.DataFrame([
    {
        "корпус": "combine_balanced",
        "состав": "Dusha (crowd+podcast), 4 эмоции, сбалансирован",
        "train (LMDB)": "68 203",
        "test (LMDB)": "6 392",
        "зачем": "Основной аудио-корпус: сбалансированные классы, mix acted+real-life",
        "модели": "CNN/CNN-BiLSTM, HuBERT+RuBERT late fusion, foundation-модели",
    },
    {
        "корпус": "combine_balanced_small",
        "состав": "30% от combine_balanced, пропорции классов сохранены",
        "train (LMDB)": "20 474",
        "test (LMDB)": "1 863",
        "зачем": "Быстрые прогоны и warm-start чекпоинтов",
        "модели": "wav2vec2 (warm-start)",
    },
    {
        "корпус": "dusha_resd",
        "состав": "combine_balanced + RESD (train 916 / test 224)",
        "train (LMDB)": "69 119",
        "test (LMDB)": "6 616",
        "зачем": "Расширение данных; RESD даёт 100% транскрипций и студийную сыгранную речь",
        "модели": "RuBERT, LogReg, RandomForest, SVM, openSMILE+XGBoost, fusion",
    },
])
display(CORPUS_INFO)

,корпус,состав,train (LMDB),test (LMDB),зачем,модели
0,combine_balanced,"Dusha (crowd+podcast), 4 эмоции, сбалансирован",68 203,6 392,Основной аудио-корпус: сбалансированные классы...,"CNN/CNN-BiLSTM, HuBERT+RuBERT late fusion, fou..."
1,combine_balanced_small,"30% от combine_balanced, пропорции классов сох...",20 474,1 863,Быстрые прогоны и warm-start чекпоинтов,wav2vec2 (warm-start)
2,dusha_resd,combine_balanced + RESD (train 916 / test 224),69 119,6 616,Расширение данных; RESD даёт 100% транскрипций...,"RuBERT, LogReg, RandomForest, SVM, openSMILE+X..."


### Экспорт сводной статистики

Сохраняем `data_statistics.csv` (таблица датасетов) и `data_statistics.json` (полный отчёт)
рядом с ноутбуком — для воспроизводимости и использования в других отчётах.

In [18]:
OUT_DIR = PROJECT_ROOT / "dusha" / "my_experiments" / "data_analise"
OUT_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "datasets": [dict(r) for r in summary.to_dict("records")],
    "lmdb": lmdb_df.to_dict("records"),
}
if sh_tr and sh_te:
    report["train_test"] = {
        "emotion_shares": {"train": sh_tr, "test": sh_te},
        "duration_mean": {"train": float(d_tr.mean()), "test": float(d_te.mean())},
    }

json_out = OUT_DIR / "data_statistics.json"
csv_out = OUT_DIR / "data_statistics.csv"
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
summary.to_csv(csv_out, index=False, encoding="utf-8-sig")

print(f"CSV:  {csv_out}")
print(f"JSON: {json_out}")

CSV:  /home/natlis/PycharmProjects/dusha_new/dusha/my_experiments/data_analise/data_statistics.csv
JSON: /home/natlis/PycharmProjects/dusha_new/dusha/my_experiments/data_analise/data_statistics.json
